> **AI-generated.** An AI assistant drafted this page, and no maintainer has reviewed it yet. Please report errors on the issue tracker.

# Terms

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TARPS-group/prob-pipe/blob/dev/overhaul/docs/user_guide/01_terms.ipynb)

Every object that ProbPipe computes with is a **term**, such as an array, a record of named fields, or a distribution.
Every term carries three things:

- **a label:** the name a reader sees for it;
- **a spec:** the declaration of its kind and its structure;
- **its stored data:** the array, mapping, or object that `.raw()` returns.

This chapter shows these three things on small examples, together with a term's immutability, its batch form, and its provenance, and every later chapter relies on them.

In [1]:
# On Google Colab, install ProbPipe; elsewhere this cell does nothing.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install --quiet "probpipe-core @ git+https://github.com/TARPS-group/prob-pipe.git@dev/overhaul"

In [2]:
import jax
import jax.numpy as jnp

from probpipe import (
    ApplicabilityError,
    DistributionBatch,
    Gamma,
    Normal,
    NumericArray,
    NumericArrayBatch,
    NumericArraySpec,
    NumericRecordBatch,
    Record,
    function,
    mean,
    positive,
    sample,
    workflow_run,
)

## What every term carries

We build one term of each of four kinds.
The first argument of each constructor is the label, and `@function` takes the label from the name of the Python function:

In [3]:
weights = NumericArray("weights", jnp.array([0.2, 0.5, 0.3]))
site = Record("site", count=jnp.array([3, 5, 2]), area=1.5)
theta = Normal("theta", 0.0, 1.0)


@function
def rescale(x: jax.Array) -> jax.Array:
    # Divide by the total, so the entries sum to one.
    return x / jnp.sum(x)


for term in [weights, site, theta, rescale]:
    print(f"kind: {type(term).__name__}, label: {term.label!r}")

kind: NumericArray, label: 'weights'
kind: NumericRecord, label: 'site'
kind: Normal, label: 'theta'
kind: Function, label: 'rescale'


`Record` returned a `NumericRecord`, which is the record class whose fields are all numeric arrays, as [Values and records](02_values_and_records.ipynb) explains.

`.raw()` returns the stored data in the form that other libraries accept, such as the JAX array of `weights` or the TensorFlow Probability distribution that `theta` stores:

In [4]:
print("stored data of weights:", weights.raw())
print("stored data of site:", site.raw())
print("stored data of theta:", theta.raw())
print("name of the Python function that rescale stores:", rescale.raw().__name__)

stored data of weights: [0.2 0.5 0.3]
stored data of site: {'count': Array([3, 5, 2], dtype=int32), 'area': Array(1.5, dtype=float32, weak_type=True)}
stored data of theta: tfp.distributions.Normal("Normal", batch_shape=[], event_shape=[], dtype=float32)
name of the Python function that rescale stores: rescale


## Labels

The label names a term in printed output and in the record of how each result was computed.
We set it at construction, and `with_label` returns a copy under a new label:

In [5]:
prior = theta.with_label("prior")

print("label of the copy:", prior.label)
print("label of the original:", theta.label)

label of the copy: prior
label of the original: theta


An operation labels its result from what produced it.
The result of a function takes the function's label, and the mean of a distribution takes the distribution's label:

In [6]:
print("label of rescale(weights):", rescale(weights).label)
print("label of the mean of theta:", mean(theta).label)

label of rescale(weights): rescale
label of the mean of theta: theta


## Immutability

A term cannot be changed after construction, so assigning to one of its attributes raises `AttributeError`:

In [7]:
try:
    weights.label = "w"
except AttributeError as error:
    print("assigning a label raised:", error)

assigning a label raised: NumericArray is immutable


A change therefore returns a new term and leaves the original as it was, as `with_label` did above.
[Values and records](02_values_and_records.ipynb) shows the changes to a record's fields, which return new records in the same way.

A term stores the array it is given, so writing into a NumPy array after construction changes every term that stores it.
To change the values, build a new array and a new term from it.

A term's **annotations** are the one part that can be written after construction.
They form a mapping in which diagnostics record their results, as `add_mcmc_diagnostics` records R-hat and effective sample sizes in [Checking results](07_checking_results.ipynb).

## Specs

A term's spec declares its kind and its structure, and `.spec` returns it.
Each kind has its own spec class:

- `NumericArraySpec`: the shape, the dtype, and the support of an array;
- `RecordSpec`: the spec of each field of a record, and `NumericRecordSpec` when every field is an array;
- `OpaqueSpec`: the Python type of a non-numeric value;
- `DistributionSpec`: the declaration of one draw of a distribution, which `event_spec` returns;
- `FunctionSpec`: the declarations of a function's inputs and of its output;
- `BatchSpec`: the spec of each element of a batch and the size of each level.

In [8]:
print("spec of weights:", weights.spec)
print("spec of site:", site.spec)
print("declaration of one draw of theta:", theta.event_spec)
print("spec of rescale:", rescale.spec)

spec of weights: NumericArraySpec(shape=(3,), dtype=float32)
spec of site: NumericRecordSpec(count=(3,), area=())
declaration of one draw of theta: OutputSpec(theta=NumericArraySpec(shape=(), dtype=float32, support=real))
spec of rescale: FunctionSpec()


A draw of `theta` is a scalar under the component `theta`, and its support `real` is the set of values a draw can take.
`rescale` declares neither its inputs nor its output, so its spec is empty, and a call accepts the arguments that its Python body accepts.

Construction checks the value against a spec passed as `spec=`.
Here the spec declares three positive weights, and the array holds two:

In [9]:
three_positive_weights = NumericArraySpec((3,), support=positive)

try:
    NumericArray("weights", jnp.array([0.4, 0.6]), spec=three_positive_weights)
except ValueError as error:
    print("construction raised:", error)

construction raised: the array does not satisfy its declaration: shape (2,) and dtype float32 against NumericArraySpec(shape=(3,), support=positive)


Construction checks the shape and the dtype.
The support is checked by the operations that depend on it, such as `convert` in [Converting and updating distributions](08_conversion_and_updating.ipynb), which refuses a family whose support differs from the distribution's.

A spec can name a dimension in place of a fixed size.
Each value binds the name to its own size, and a name used twice takes one size, so this spec declares a square matrix of any size:

In [10]:
square = NumericArraySpec(("n", "n"))

print("named dimensions:", square.free_dims)
print("a 2 x 2 array matches:", square.is_valid(jnp.eye(2)))
print("a 3 x 3 array matches:", square.is_valid(jnp.eye(3)))
print("a 2 x 3 array matches:", square.is_valid(jnp.ones((2, 3))))
print("the spec with n bound to 4:", square.with_dim_sizes(n=4))

named dimensions: frozenset({'n'})
a 2 x 2 array matches: True


a 3 x 3 array matches: True
a 2 x 3 array matches: False
the spec with n bound to 4: NumericArraySpec(shape=(4, 4))


A function can declare its inputs and its output.
`@function` takes `input_spec`, which maps each parameter to its spec, and `output_spec`, which declares the result.
A call then checks its arguments against the declaration and raises `ApplicabilityError` for an argument that does not match:

In [11]:
@function(input_spec={"counts": NumericArraySpec(("n",))}, output_spec=NumericArraySpec(()))
def total(counts: jax.Array) -> jax.Array:
    return jnp.sum(counts)


print("spec of total:", total.spec)
print("total of three counts:", total(jnp.array([3, 5, 2])).raw())

try:
    total(jnp.ones((2, 3)))
except ApplicabilityError as error:
    print("a matrix of counts raised:", error)

spec of total: FunctionSpec(
    input_spec=InputSpec(counts=NumericArraySpec(shape=('n',))),
    output_spec=OutputSpec(total=NumericArraySpec(shape=())),
)


total of three counts: 10
a matrix of counts raised: Function 'total' input/counts has rank 2, expected rank 1 from shape ('n',)


The output declaration names the component `total`, and a distribution's `event_spec` names the components of a draw in the same way.
A function's result and a distribution's draw therefore share one kind of declaration, which [Joint models and conditional distributions](04_joint_models.ipynb) uses to compose a model from its parts.

## Batches

Every kind of term has a batch form, which holds several terms of that kind along named axes called **levels**.
`NumericArrayBatch(label, values, level_names)` builds a batch of arrays, and the leading axes of `values` are the levels.
Here the heights of three people at each of two sites form a batch on the levels `site` and `person`:

In [12]:
heights = NumericArrayBatch(
    "heights",
    jnp.array([[1.71, 1.64, 1.80], [1.55, 1.62, 1.70]]),
    ("site", "person"),
)

print("batch:", heights)
print("spec of the batch:", heights.spec)

batch: NumericArrayBatch(
    'heights',
    levels={'site': 2, 'person': 3},
    element_spec=NumericArraySpec(shape=(), dtype=float32),
)
spec of the batch: BatchSpec(element_spec=NumericArraySpec(shape=(), dtype=float32), levels={'site': 2, 'person': 3})


Each element is one height, so the element spec declares a scalar, and the levels hold the six repetitions.
A batch of six heights is therefore a different term from one array of six heights: a function called on the batch evaluates at each height, as [Functions on values and distributions](05_functions_and_lifting.ipynb) shows.

Indexing by position selects along the levels in order, and `at_levels` selects by level name.
A selection that keeps a level returns a smaller batch, and a selection that fixes every level returns an element, labeled by its position:

In [13]:
print("the first site:", repr(heights[0]))
print("the second person at the first site:", repr(heights[0, 1]))
print("the second person at every site:", repr(heights.at_levels(person=1)))
print("shape of the stored array:", heights.raw().shape)

the first site: NumericArrayBatch(
    'heights[site=0]',
    levels={'person': 3},
    element_spec=NumericArraySpec(shape=(), dtype=float32),
)
the second person at the first site: NumericArray('heights[site=0, person=1]', shape=(), dtype=float32)
the second person at every site: NumericArrayBatch(
    'heights[person=1]',
    levels={'site': 2},
    element_spec=NumericArraySpec(shape=(), dtype=float32),
)
shape of the stored array: (2, 3)


`.raw()` returns the stored array, whose leading axes are the levels.

A batch of records stores one column per field.
Indexing by position returns one record, and indexing by a field name returns that field's column as a batch:

In [14]:
sites = NumericRecordBatch(
    "sites",
    {"count": jnp.array([[3, 5, 2], [4, 1, 2]]), "area": jnp.array([1.5, 2.0])},
    "site",
)

print("batch of records:", repr(sites))
print("the second site:", repr(sites[1]))
print("fields of the second site:", sites[1].raw())
print("the area column:", repr(sites["area"]))

batch of records: NumericRecordBatch('sites', levels={'site': 2}, fields=('count', 'area'))
the second site: NumericRecord('sites[site=1]', fields=('count', 'area'))
fields of the second site: {'count': Array([4, 1, 2], dtype=int32), 'area': Array(2., dtype=float32)}
the area column: NumericArrayBatch('area', levels={'site': 2}, element_spec=NumericArraySpec(shape=()))


Most batches come from operations.
`sample` returns its draws as a batch on the level `sample`, and the draws of a joint distribution form a batch of records.
`workflow_run(seed=...)` makes the draws reproducible, as [How a result was computed](09_how_results_are_computed.ipynb) explains:

In [15]:
model = Normal("mu", 0.0, 1.0) * Gamma("sigma", 2.0, 1.0)

with workflow_run(seed=0):
    draws_of_theta = sample(theta, sample_shape=5)
    draws_of_model = sample(model, sample_shape=4)

print("draws of theta:", repr(draws_of_theta))
print("draws of the model:", repr(draws_of_model))

draws of theta: NumericArrayBatch(
    'theta',
    levels={'sample': 5},
    element_spec=NumericArraySpec(shape=(), dtype=float32, support=real),
)
draws of the model: NumericRecordBatch('mu·sigma', levels={'sample': 4}, fields=('mu', 'sigma'))


Distributions and functions have batch forms as well.
A `DistributionBatch` holds distributions whose draws share one declaration, and an operation on it returns a batch of results, one per distribution:

In [16]:
growth = DistributionBatch(
    "growth", [Normal("growth", 0.0, 1.0), Normal("growth", 1.0, 2.0)], "region"
)

print("the second distribution:", repr(growth[1]))
print("mean of each distribution:", mean(growth).raw())

the second distribution: Normal(
    'growth[region=1]',
    loc=1.0,
    scale=2.0,
    event_spec=OutputSpec(growth=NumericArraySpec(shape=(), dtype=float32, support=real)),
)
mean of each distribution: [0. 1.]


[Functions on values and distributions](05_functions_and_lifting.ipynb) calls a function on a batch of inputs and a distribution together, which returns a batch of distributions.

## Provenance

A term computed by an operation records in its **provenance** the operation and the terms that the operation used.
A term that we construct directly has no provenance:

In [17]:
print("provenance of rescale(weights):", rescale(weights).provenance)
print("provenance of the mean of theta:", mean(theta).provenance)
print("provenance of weights:", weights.provenance)

provenance of rescale(weights): Provenance('workflow.rescale', parents=[rescale, weights])
provenance of the mean of theta: Provenance('workflow.mean', parents=[mean, theta])
provenance of weights: None


[How a result was computed](09_how_results_are_computed.ipynb) follows the provenance of a posterior back to the data it was conditioned on.

## See also

- [Labels and provenance](../api/provenance.md): the API reference for `TrackedTerm` and `Provenance`.
- [Declarations](../api/declarations.md): the spec classes and the supports, such as `positive`.
- [Values and records](../api/values.md): the API reference for the values and their batches, such as `NumericArrayBatch` and `RecordBatch`.
- [Values and records](02_values_and_records.ipynb): records, their fields, and the values a field can hold.
- [Distributions](03_distributions.ipynb): the declaration of a draw, and the operations on a distribution.
- [Functions on values and distributions](05_functions_and_lifting.ipynb): functions called on values, distributions, and batches.